In [1]:
# Load skill importance ratings and the skill-to-bundle mapping.
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/GenAI-human-skill-adaptation-dev/Occupational-level")


skill_importance = pd.read_csv(
    ROOT / "01_Source_Data_Preparation" / "Output" / "Occupational_Skill_Importance_Long.csv"
)

bundle_spec = pd.read_csv(
    ROOT / "02_Skill_Bundle_Specification" / "Output" / "Skill_Bundle_Specification.csv"
)

skill_importance.head()

,O*NET-SOC Code,Element ID,Element Name,Scale ID,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source,soc6
0,11-1011.00,2.A.1.a,Reading Comprehension,IM,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst,11-1011
1,11-1011.00,2.A.1.b,Active Listening,IM,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst,11-1011
2,11-1011.00,2.A.1.c,Writing,IM,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst,11-1011
3,11-1011.00,2.A.1.d,Speaking,IM,4.25,8,0.1637,3.9292,4.5708,N,NaN,08/2023,Analyst,11-1011
4,11-1011.00,2.A.1.e,Mathematics,IM,3.25,8,0.2500,2.7600,3.7400,N,NaN,08/2023,Analyst,11-1011


In [2]:
# Keep the selected skill ratings and attach their bundle labels.
skill_bundles = skill_importance[
    ["O*NET-SOC Code", "soc6", "Element ID", "Data Value"]
].merge(
    bundle_spec[["Element ID", "Bundle"]],
    on="Element ID", how="inner", validate="many_to_one"
)

In [3]:
skill_bundles.head()

,O*NET-SOC Code,soc6,Element ID,Data Value,Bundle
0,11-1011.00,11-1011,2.A.1.a,4.12,Communication and interpersonal skills
1,11-1011.00,11-1011,2.A.1.b,4.00,Communication and interpersonal skills
2,11-1011.00,11-1011,2.A.1.c,4.12,Communication and interpersonal skills
3,11-1011.00,11-1011,2.A.1.d,4.25,Communication and interpersonal skills
4,11-1011.00,11-1011,2.A.1.e,3.25,Technical and analytical skills


In [4]:
# Calculate a bundle score by averaging its member skills within each detailed occupation.
detailed_bundle_scores = (
    skill_bundles.groupby(["O*NET-SOC Code", "soc6", "Bundle"], as_index=False)
    .agg(bundle_score=("Data Value", "mean"))
)

In [5]:
detailed_bundle_scores.head()

,O*NET-SOC Code,soc6,Bundle,bundle_score
0,11-1011.00,11-1011,Communication and interpersonal skills,3.948000
1,11-1011.00,11-1011,Complex problem solving and systems analysis,4.250000
2,11-1011.00,11-1011,Critical thinking and evaluation,3.852000
3,11-1011.00,11-1011,Learning and time management,3.623333
4,11-1011.00,11-1011,Technical and analytical skills,2.165000


In [6]:
# Average detailed occupation scores to obtain one score per SOC6 and bundle.
soc6_bundle_scores = (
    detailed_bundle_scores.groupby(["soc6", "Bundle"], as_index=False)
    .agg(bundle_score=("bundle_score", "mean"))
)

In [7]:
soc6_bundle_scores.head()

,soc6,Bundle,bundle_score
0,11-1011,Communication and interpersonal skills,3.836500
1,11-1011,Complex problem solving and systems analysis,4.095000
2,11-1011,Critical thinking and evaluation,3.677000
3,11-1011,Learning and time management,3.563333
4,11-1011,Technical and analytical skills,2.135833


In [8]:
# Reshape bundle scores into one row per SOC6 and one column per bundle.
occupational_bundle_scores = (
    soc6_bundle_scores.pivot(index="soc6", columns="Bundle", values="bundle_score")
    .rename_axis(columns=None)
    .reset_index()
)

In [9]:
# Save occupational bundle scores for Phase 04.

occupational_bundle_scores.to_csv(
    ROOT / "03_Skill_Bundle_Construction" / "Output" / "Occupational_Skill_Bundle_Scores.csv",
    index=False
)